In [1]:
import numpy as np
import pandas as pd
from pykalman import KalmanFilter
import vectorbt as vbt
import yfinance as yf

### JPMORGAN vs Bank of America

In [3]:

pairs = ["JPM", "BAC"]
df3 = yf.download(pairs, start="2015-05-05", end="2025-05-05", progress=False)

try:
  stock_a = df3["Close"]["JPM"]
  stock_b = df3["Close"]["BAC"]
except (KeyError, TypeError):
  stock_a = df3.xs("Close", level=0, axis=1)["JPM"]
  stock_b = df3.xs("Close", level=0, axis=1)["BAC"]

price_data = pd.DataFrame({"JPM": stock_a, "BAC": stock_b}).dropna()
s_a = price_data["JPM"]
s_b = price_data["BAC"]


delta = 1e-3
trans_cov = delta / (1 - delta) * np.eye(2)
obs_mat = np.vstack([s_b.values, np.ones(len(s_b))]).T[:, np.newaxis, :]

kf = KalmanFilter(
    n_dim_obs=1,
    n_dim_state=2,
    initial_state_mean=[0, 0],
    initial_state_covariance=np.ones((2, 2)),
    transition_matrices=np.eye(2),
    observation_matrices=obs_mat,
    observation_covariance=1.0,
    transition_covariance=trans_cov,
)

state_means, _ = kf.filter(s_a.values)
beta = state_means[:, 0]
alpha = state_means[:, 1]


spread = s_a - (beta * s_b + alpha)
spread_series = pd.Series(spread, index=s_a.index)

window = 60  
rolling_mean = spread_series.rolling(window=window).mean()
rolling_std = spread_series.rolling(window=window).std()
zscore = (spread_series - rolling_mean) / rolling_std

entry_threshold = 2.5
stop_loss_threshold = 4.0


long_spread = zscore < -entry_threshold
short_spread = zscore > entry_threshold
exit_cond = (zscore.abs() <= 0.2) | (zscore.abs() >= stop_loss_threshold)

entries = pd.DataFrame({"JPM": long_spread, "BAC": short_spread}, index=s_a.index)

short_entries = pd.DataFrame(
    {"JPM": short_spread, "BAC": long_spread}, index=s_a.index
)

exits = pd.DataFrame({"JPM": exit_cond, "BAC": exit_cond}, index=s_a.index)

short_exits = pd.DataFrame({"JPM": exit_cond, "BAC": exit_cond}, index=s_a.index)

# 5. Multi-Asset Vectorbt Portfolio Simulation
backtest = vbt.Portfolio.from_signals(
    close=price_data,
    entries=entries,
    exits=exits,
    short_entries=short_entries,
    short_exits=short_exits,
    init_cash=100000.0,
    fees=0.0002, 
    cash_sharing=True,
    group_by=True,
    freq="1D",
)

print(backtest.stats())
backtest.plot().show()

Start                            2015-05-05 00:00:00
End                              2025-05-02 00:00:00
Period                            2515 days 00:00:00
Start Value                                 100000.0
End Value                              265408.125748
Total Return [%]                          165.408126
Benchmark Return [%]                      313.172586
Max Gross Exposure [%]                    386.525209
Total Fees Paid                          6102.458701
Max Drawdown [%]                           18.959846
Max Drawdown Duration              466 days 00:00:00
Total Trades                                      67
Total Closed Trades                               67
Total Open Trades                                  0
Open Trade PnL                                   0.0
Win Rate [%]                               64.179104
Best Trade [%]                             33.342885
Worst Trade [%]                           -12.961329
Avg Winning Trade [%]                       3.

c:\Users\Haresh L\anaconda3\Lib\site-packages\vectorbt\generic\plots_builder.py:329: UserWarning:

Subplot 'orders' does not support grouped data

c:\Users\Haresh L\anaconda3\Lib\site-packages\vectorbt\generic\plots_builder.py:329: UserWarning:

Subplot 'trade_pnl' does not support grouped data



### Using a shorter window

In [10]:

pairs = ["JPM", "BAC"]
df3 = yf.download(pairs, start="2015-05-05", end="2025-05-05", progress=False)

try:
  stock_a = df3["Close"]["JPM"]
  stock_b = df3["Close"]["BAC"]
except (KeyError, TypeError):
  stock_a = df3.xs("Close", level=0, axis=1)["JPM"]
  stock_b = df3.xs("Close", level=0, axis=1)["BAC"]

price_data = pd.DataFrame({"JPM": stock_a, "BAC": stock_b}).dropna()
s_a = price_data["JPM"]
s_b = price_data["BAC"]


delta = 1e-3
trans_cov = delta / (1 - delta) * np.eye(2)
obs_mat = np.vstack([s_b.values, np.ones(len(s_b))]).T[:, np.newaxis, :]

kf = KalmanFilter(
    n_dim_obs=1,
    n_dim_state=2,
    initial_state_mean=[0, 0],
    initial_state_covariance=np.ones((2, 2)),
    transition_matrices=np.eye(2),
    observation_matrices=obs_mat,
    observation_covariance=1.0,
    transition_covariance=trans_cov,
)

state_means, _ = kf.filter(s_a.values)
beta = state_means[:, 0]
alpha = state_means[:, 1]


spread = s_a - (beta * s_b + alpha)
spread_series = pd.Series(spread, index=s_a.index)

window = 30  
rolling_mean = spread_series.rolling(window=window).mean()
rolling_std = spread_series.rolling(window=window).std()
zscore = (spread_series - rolling_mean) / rolling_std

entry_threshold = 2.5
stop_loss_threshold = 4.0


long_spread = zscore < -entry_threshold
short_spread = zscore > entry_threshold
exit_cond = (zscore.abs() <= 0.2) | (zscore.abs() >= stop_loss_threshold)

entries = pd.DataFrame({"JPM": long_spread, "BAC": short_spread}, index=s_a.index)

short_entries = pd.DataFrame(
    {"JPM": short_spread, "BAC": long_spread}, index=s_a.index
)

exits = pd.DataFrame({"JPM": exit_cond, "BAC": exit_cond}, index=s_a.index)

short_exits = pd.DataFrame({"JPM": exit_cond, "BAC": exit_cond}, index=s_a.index)

# 5. Multi-Asset Vectorbt Portfolio Simulation
backtest = vbt.Portfolio.from_signals(
    close=price_data,
    entries=entries,
    exits=exits,
    short_entries=short_entries,
    short_exits=short_exits,
    init_cash=100000.0,
    fees=0.0002, 
    cash_sharing=True,
    group_by=True,
    freq="1D",
)

print(backtest.stats())
backtest.plot().show()

Start                            2015-05-05 00:00:00
End                              2025-05-02 00:00:00
Period                            2515 days 00:00:00
Start Value                                 100000.0
End Value                               191495.53095
Total Return [%]                           91.495531
Benchmark Return [%]                      313.172586
Max Gross Exposure [%]                     444.62481
Total Fees Paid                          4079.560078
Max Drawdown [%]                           24.176896
Max Drawdown Duration              670 days 00:00:00
Total Trades                                      55
Total Closed Trades                               55
Total Open Trades                                  0
Open Trade PnL                                   0.0
Win Rate [%]                               67.272727
Best Trade [%]                             23.769262
Worst Trade [%]                           -17.693418
Avg Winning Trade [%]                       3.

c:\Users\Haresh L\anaconda3\Lib\site-packages\vectorbt\generic\plots_builder.py:329: UserWarning:

Subplot 'orders' does not support grouped data

c:\Users\Haresh L\anaconda3\Lib\site-packages\vectorbt\generic\plots_builder.py:329: UserWarning:

Subplot 'trade_pnl' does not support grouped data

